# Resolution concordance exploration

Interactive browsing of `resolution_concordance_1626_1630` (refreshed 2026-09-23) — one row per
enriched resolution (19,120 total, 1626-01-01 through end of 1630), carrying `day_status`, the
resolved HTR session, and best-effort paragraph placement. Also applies PLAN.md's 'separated'
acceptance criterion per resolution (not just the day-level aggregate
`scripts/metrics_separation_span_table.py` reports), and spot-checks placed resolutions against
the HTR paragraph text they were mapped to.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / 'data_manifest.toml').exists()),
    Path.cwd(),
)
for import_root in (PROJECT_ROOT, PROJECT_ROOT / 'src'):
    if import_root.exists() and str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

import pandas as pd

from data_io import load
from scripts.metrics_separation_span_table import MAX_EXTENT_PARAGRAPHS, CEILING_SEPARABLE, count_separated_per_day

pd.set_option('display.max_colwidth', 120)

## 1. Load the concordance table

In [ ]:
concordance = load('resolution_concordance_1626_1630')
print(concordance.shape)
concordance.head(3)

## 2. Day-status breakdown

In [ ]:
concordance['day_status'].value_counts()

## 3. Paragraph placement coverage

`paragraph_start_index`/`paragraph_end_index` are set only when `paragraph_prediction_status ==
'predicted'`; everything else (mostly `missing_htr` days) is `NaN` — these are the unaligned
resolutions. `[start, end)` is a half-open range over *positions in the per-day paragraph stream*
(see section 5 — not the same as the `para_index` field on individual paragraph_axis records).

In [ ]:
has_placement = concordance['paragraph_start_index'].notna()
print(f"placed:    {has_placement.sum()} / {len(concordance)}")
print(f"unaligned: {(~has_placement).sum()} / {len(concordance)}")
concordance.loc[has_placement, 'paragraph_prediction_status'].value_counts(dropna=False)

## 4. Per-resolution 'separated' flag

A resolution is **separated** iff its start paragraph is shared with no other resolution on its
`enriched_date`, and its extent (`paragraph_end_index - paragraph_start_index + 1`) is
`<= MAX_EXTENT_PARAGRAPHS` (3) — PLAN.md's Global-primary criterion. Logic mirrors
`count_separated_per_day` in `scripts/metrics_separation_span_table.py`; the validation cell below
cross-checks the day-level total against that script's own output so the two never silently drift
apart.

In [ ]:
placed = concordance.loc[has_placement, ['enriched_date', 'resolution_index', 'enriched_id', 'paragraph_start_index', 'paragraph_end_index']].copy()
placed['extent'] = (placed['paragraph_end_index'] - placed['paragraph_start_index'] + 1).astype(int)

start_share = placed.groupby(['enriched_date', 'paragraph_start_index'])['resolution_index'].transform('size')
placed['separated'] = (start_share == 1) & (placed['extent'] <= MAX_EXTENT_PARAGRAPHS)

separated_ids = set(placed.loc[placed['separated'], 'enriched_id'])
concordance['separated'] = concordance['enriched_id'].isin(separated_ids)

n_separated = int(concordance['separated'].sum())
print(f"separated: {n_separated} / {len(concordance)} all resolutions")
print(f"separated: {n_separated} / {CEILING_SEPARABLE} ceiling ({n_separated / CEILING_SEPARABLE:.1%})")

In [ ]:
per_day_check = count_separated_per_day(concordance)
assert per_day_check['separated_count'].sum() == n_separated, "notebook separated-flag logic has drifted from metrics_separation_span_table.py"
print("OK — matches scripts/metrics_separation_span_table.py's per-day total")

## 5. Browse: separated vs. unaligned resolutions

Spot-check a few placed resolutions' enriched text against the HTR paragraph text they were mapped
to. `paragraph_start_index`/`paragraph_end_index` are positions in the **per-day paragraph stream**
exactly as `scripts/s4_corpus_paragraph_predictions.py` builds it — *not* a filter on
`paragraph_axis_1626_1630`'s own `para_index` field, which resets to 0 at the start of every HTR
session, so multiple sessions sharing a date can share `para_index` values. Reproduces
`axis_for_date`: prefer the concordance-resolved session's own paragraphs
(`resolved_session_id`, grouped from `paragraph_axis_1626_1630` by the session portion of
`flat_id`) over the same-calendar-date paragraphs, falling back to the latter only when no
resolved session (or an empty one) is recorded — see that function's docstring for why.

In [ ]:
axis_records = load('paragraph_axis_1626_1630')


def session_of(flat_id: str) -> str:
    return str(flat_id).split('-resolution-', 1)[0]


axis_by_date: dict[str, list[dict]] = {}
axis_by_session: dict[str, list[dict]] = {}
for record in axis_records:
    axis_by_date.setdefault(str(record['date']), []).append(record)
    axis_by_session.setdefault(session_of(record['flat_id']), []).append(record)


def axis_for_row(row: pd.Series) -> list[dict]:
    session_id = row.get('resolved_session_id')
    if isinstance(session_id, str) and session_id:
        resolved = axis_by_session.get(session_id)
        if resolved:
            return resolved
    return axis_by_date.get(row['enriched_date'], [])


def show_resolution(enriched_id: str) -> None:
    row = concordance.loc[concordance['enriched_id'] == enriched_id].iloc[0]
    print(f"{enriched_id}  status={row['day_status']}  separated={row.get('separated')}")
    print(f"enriched text: {str(row['text'])[:400]}")
    if pd.isna(row['paragraph_start_index']):
        print("(no paragraph placement)")
        return
    stream = axis_for_row(row)
    start, end = int(row['paragraph_start_index']), int(row['paragraph_end_index'])
    if not stream or end > len(stream):
        print(f"(paragraph stream unavailable or shorter than expected: {len(stream)} paragraphs, wanted [{start}, {end}))")
        return
    for p in stream[start:end]:
        print(f"  [{p['axis_id']}] {p['text'][:300]}")

In [ ]:
print("=== separated examples ===")
for eid in concordance.loc[concordance['separated'], 'enriched_id'].sample(2, random_state=1):
    show_resolution(eid)
    print()

In [ ]:
print("=== unaligned examples ===")
for eid in concordance.loc[~has_placement, 'enriched_id'].sample(2, random_state=1):
    show_resolution(eid)
    print()